In [1]:
!uv add pinecone-text pinecone-client pinecone-notebooks python-dotenv langchain langchain-community langchain-huggingface

Resolved 82 packages in 3ms
Checked 81 packages in 6ms


In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from pinecone import Pinecone,ServerlessSpec

index_name = "hybrid-search-langchain-pinecone"
# Initializing pinecone client
pc = Pinecone()

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, #Using sentence Transformer
        metric='dotproduct', # As we know sparse matrix is supported for dotproduct only and not for cosine
        spec=ServerlessSpec(cloud='aws',region='us-east-1')

    )

In [5]:
index = pc.Index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-8ulyyyt.svc.aped-4627-b74a.pinecone.io')

In [6]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
embeddings = HuggingFaceEndpointEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')
embeddings

d:\tutorial\chatbot\Hybrid Search Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFaceEndpointEmbeddings(client=<InferenceClient(model='sentence-transformers/all-MiniLM-L6-v2', timeout=None)>, async_client=<InferenceClient(model='sentence-transformers/all-MiniLM-L6-v2', timeout=None)>, model='sentence-transformers/all-MiniLM-L6-v2', provider=None, repo_id='sentence-transformers/all-MiniLM-L6-v2', task='feature-extraction', model_kwargs=None, huggingfacehub_api_token='[REDACTED_HF_TOKEN]')

In [7]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [8]:
sentences = [
    "The capital of France is Paris, known for landmarks like the Eiffel Tower.",
    "Water boils at 100 degrees Celsius under standard atmospheric pressure.",
    "The human brain contains approximately 86 billion neurons.",
    "The Great Wall of China stretches over 13,000 miles.",

    "Retrieval-Augmented Generation combines vector search with large language models.",
    "Transformers rely on self-attention mechanisms to process sequential data.",
    "Python is widely used for machine learning due to libraries like TensorFlow and PyTorch.",
    "A REST API typically uses HTTP methods such as GET, POST, PUT, and DELETE.",

    "Apple released a new product this year.",
    "Apple is a popular fruit rich in fiber and vitamins.",
    "The bank approved the loan request quickly.",
    "He sat by the bank of the river to relax.",

    "The Amazon rainforest produces about 20% of the world's oxygen and spans multiple countries.",
    "The theory of relativity revolutionized our understanding of space, time, and gravity.",

]
#tfidf values on these sentences
bm25_encoder.fit(sentences)

# store the values in a json file
bm25_encoder.dump('bm25_values.json')

#load to our BM25Encoder object
bm25_encoder = BM25Encoder().load('bm25_values.json')


100%|██████████| 14/14 [00:00<00:00, 308.29it/s]


In [9]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)

In [10]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEndpointEmbeddings(client=<InferenceClient(model='sentence-transformers/all-MiniLM-L6-v2', timeout=None)>, async_client=<InferenceClient(model='sentence-transformers/all-MiniLM-L6-v2', timeout=None)>, model='sentence-transformers/all-MiniLM-L6-v2', provider=None, repo_id='sentence-transformers/all-MiniLM-L6-v2', task='feature-extraction', model_kwargs=None, huggingfacehub_api_token='[REDACTED_HF_TOKEN]'), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x0000022E79C4C830>, index=Index(host='https://hybrid-search-langchain-pinecone-8ulyyyt.svc.aped-4627-b74a.pinecone.io'))

In [11]:
retriever.add_texts(texts=sentences)

100%|██████████| 1/1 [00:03<00:00,  3.50s/it]


In [12]:
retriever.invoke("What is the length of Great Wall of China")

[Document(metadata={'score': 0.575355768}, page_content='The Great Wall of China stretches over 13,000 miles.'),
 Document(metadata={'score': 0.081205368}, page_content='The human brain contains approximately 86 billion neurons.'),
 Document(metadata={'score': 0.0797963142}, page_content='The capital of France is Paris, known for landmarks like the Eiffel Tower.'),
 Document(metadata={'score': 0.0368230343}, page_content='The bank approved the loan request quickly.')]

In [13]:
retriever.invoke("Which Popular fruit gives us vitamin and Which country capital is France?")

[Document(metadata={'score': 0.328864455}, page_content='Apple is a popular fruit rich in fiber and vitamins.'),
 Document(metadata={'score': 0.20126003}, page_content='The capital of France is Paris, known for landmarks like the Eiffel Tower.'),
 Document(metadata={'score': 0.12657772}, page_content="The Amazon rainforest produces about 20% of the world's oxygen and spans multiple countries."),
 Document(metadata={'score': 0.0785679817}, page_content='Apple released a new product this year.')]